# 08 — Paper Tables

**Project:** Causal ML for Climate Policy Effectiveness  
**Purpose:** Generate LaTeX-formatted tables for the Nexus manuscript  
**Output:** LaTeX (.tex) and CSV tables in `outputs/tables/`

---

In [1]:
import sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TABLE_DIR = PROJECT_ROOT / "outputs" / "tables"
TABLE_DIR.mkdir(parents=True, exist_ok=True)

panel = pd.read_csv(PROCESSED_DIR / "analysis_panel.csv")
print(f"Panel loaded: {panel.shape}")

Panel loaded: (7630, 74)


---
## Table 1: Summary Statistics

In [2]:
# Select key variables for summary table
summary_vars = []
var_labels = {}

# Outcome
for v, l in [("co2", "CO₂ Emissions (Mt)"), ("log_co2", "Log CO₂ Emissions"),
             ("co2_per_capita", "CO₂ per Capita (t)")]:
    if v in panel.columns:
        summary_vars.append(v)
        var_labels[v] = l

# Treatments
for v in panel.columns:
    if v.startswith("treat_"):
        summary_vars.append(v)
        var_labels[v] = v.replace("treat_", "").replace("_", " ").title()

# Covariates
for v, l in [("gdp", "GDP"), ("population", "Population"),
             ("gdp_per_capita", "GDP per Capita"),
             ("renewables_share_energy", "Renewables Share (%)"),
             ("urban_pct", "Urban Population (%)"),
             ("regulatory_quality", "Regulatory Quality")]:
    if v in panel.columns:
        summary_vars.append(v)
        var_labels[v] = l

summary = panel[summary_vars].describe().T
summary.index = [var_labels.get(v, v) for v in summary.index]
summary = summary[["count", "mean", "std", "min", "50%", "max"]]
summary.columns = ["N", "Mean", "Std. Dev.", "Min", "Median", "Max"]

print("\nTable 1: Summary Statistics")
print("=" * 75)
print(summary.round(3).to_string())

# Save as LaTeX
latex = summary.round(3).to_latex(
    caption="Summary Statistics of Key Variables",
    label="tab:summary_stats",
    position="htbp",
)
with open(TABLE_DIR / "table1_summary_stats.tex", "w", encoding="utf-8") as f:
    f.write(latex)

summary.round(3).to_csv(TABLE_DIR / "table1_summary_stats.csv")
print("\nSuccess: Table 1 saved (LaTeX + CSV)")


Table 1: Summary Statistics
                           N          Mean     Std. Dev.           Min        Median           Max
CO₂ Emissions (Mt)    7508.0  1.375440e+02  6.752860e+02  0.000000e+00  7.172000e+00  1.228904e+04
Log CO₂ Emissions     7508.0  1.867000e+00  2.853000e+00 -6.908000e+00  1.970000e+00  9.416000e+00
CO₂ per Capita (t)    7442.0  5.023000e+00  7.795000e+00  0.000000e+00  2.745000e+00  3.647910e+02
Carbon Pricing        7630.0  2.420000e-01  4.290000e-01  0.000000e+00  0.000000e+00  1.000000e+00
Renewable Mandate     7630.0  3.320000e-01  4.710000e-01  0.000000e+00  0.000000e+00  1.000000e+00
Efficiency Standard   7630.0  3.110000e-01  4.630000e-01  0.000000e+00  0.000000e+00  1.000000e+00
Fiscal Policy         7630.0  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
GDP                   5410.0  4.894066e+11  1.704830e+12  2.571720e+08  6.108450e+10  2.696602e+13
Population            7560.0  3.130214e+07  1.253051e+08  5.010000e+02  5.512436


Success: Table 1 saved (LaTeX + CSV)


---
## Table 2: Policy Coverage

In [3]:
df_policies = pd.read_csv(PROJECT_ROOT / "data" / "interim" / "policies_cleaned.csv")

# Policy type × count × country count
policy_summary = (
    df_policies
    .groupby("policy_category")
    .agg(
        n_policies=("iso3", "count"),
        n_countries=("iso3", "nunique"),
        earliest=("year_adopted", "min"),
        latest=("year_adopted", "max"),
    )
    .reset_index()
)
policy_summary.columns = ["Policy Type", "N Policies", "N Countries", "Earliest", "Latest"]

print("\nTable 2: Climate Policy Coverage")
print("=" * 65)
print(policy_summary.to_string(index=False))

latex = policy_summary.to_latex(
    index=False,
    caption="Climate Policy Coverage by Type",
    label="tab:policy_coverage",
)
with open(TABLE_DIR / "table2_policy_coverage.tex", "w") as f:
    f.write(latex)

policy_summary.to_csv(TABLE_DIR / "table2_policy_coverage.csv", index=False)
print("Success: Table 2 saved")


Table 2: Climate Policy Coverage
        Policy Type  N Policies  N Countries  Earliest  Latest
     Carbon Pricing         910           82      1944    2025
Efficiency Standard        2664           93      1965    2025
              Other        8707          194      1927    2025
  Renewable Mandate         853          147      1991    2025
Success: Table 2 saved


---
## Table 3: Main Results (Method Comparison)

In [4]:
try:
    comparison = pd.read_csv(TABLE_DIR / "method_comparison.csv")
    
    print("\nTable 3: Main Results — Average Treatment Effects")
    print("=" * 80)
    print(comparison.to_string(index=False))
    
    latex = comparison.to_latex(
        index=False,
        caption="Average Treatment Effects: OLS, DML, and Causal Forest Estimates",
        label="tab:main_results",
    )
    with open(TABLE_DIR / "table3_main_results.tex", "w") as f:
        f.write(latex)
    print("Success: Table 3 saved")
except FileNotFoundError:
    print("Warning: Run Notebook 05 first")


Table 3: Main Results — Average Treatment Effects
       Method                 Treatment  Estimate  Std. Error             95% CI  p-value    N Significant
          OLS      treat_carbon_pricing    0.1046      0.0508   [0.0049, 0.2042]   0.0397 2028         ***
          OLS   treat_renewable_mandate    0.0119      0.0432  [-0.0728, 0.0966]   0.7828 2028         NaN
          OLS treat_efficiency_standard    0.1196      0.0667  [-0.0112, 0.2504]   0.0731 2028         NaN
DML (xgboost)      treat_carbon_pricing   -0.0057      0.0090  [-0.0234, 0.0119]   0.5249 2028         NaN
DML (xgboost)   treat_renewable_mandate   -0.0146      0.0075 [-0.0292, -0.0000]   0.0494 2028         ***
DML (xgboost) treat_efficiency_standard    0.0070      0.0129  [-0.0183, 0.0323]   0.5888 2028         NaN
Causal Forest      treat_carbon_pricing   -0.0081      0.0354  [-0.0776, 0.0613]   0.8185 2028         NaN
Success: Table 3 saved


---
## Table 4: Robustness Summary

In [5]:
try:
    alt_specs = pd.read_csv(TABLE_DIR / "alternative_specifications.csv")
    
    print("\nTable 4: Robustness — Alternative Specifications")
    print("=" * 70)
    print(alt_specs.to_string(index=False))
    
    latex = alt_specs.to_latex(
        index=False,
        caption="Robustness: Alternative Covariate Specifications",
        label="tab:robustness",
    )
    with open(TABLE_DIR / "table4_robustness.tex", "w") as f:
        f.write(latex)
    print("Success: Table 4 saved")
except FileNotFoundError:
    print("Warning: Run Notebook 06 first")


Table 4: Robustness — Alternative Specifications
specification  n_covariates  estimate  ci_lower  ci_upper  std_error  n_obs
      Minimal             2 -0.219579 -0.339342 -0.099817   0.061104   6845
     Baseline             4 -0.193926 -0.307879 -0.079974   0.058140   6655
     Extended             6  0.104574  0.004914  0.204234   0.050848   2028
Success: Table 4 saved


In [6]:
print("\n" + "=" * 60)
print("Success: All paper tables generated!")
print("=" * 60)

# List all generated tables
table_files = sorted(TABLE_DIR.glob("*.tex"))
csv_files = sorted(TABLE_DIR.glob("*.csv"))

print(f"\nLaTeX tables ({len(table_files)}):")
for f in table_files:
    print(f"  - {f.name}")

print(f"\nCSV tables ({len(csv_files)}):")
for f in csv_files:
    print(f"  - {f.name}")

print("\nSuccess: Pipeline complete! Ready for manuscript writing.")


Success: All paper tables generated!

LaTeX tables (4):
  - table1_summary_stats.tex
  - table2_policy_coverage.tex
  - table3_main_results.tex
  - table4_robustness.tex

CSV tables (12):
  - alternative_specifications.csv
  - event_study_pretrend_summary.csv
  - event_study_treat_carbon_pricing.csv
  - event_study_treat_efficiency_standard.csv
  - event_study_treat_renewable_mandate.csv
  - method_comparison.csv
  - placebo_test_outcome.csv
  - sector_estimates.csv
  - summary_statistics.csv
  - table1_summary_stats.csv
  - table2_policy_coverage.csv
  - table_sensitivity.csv

Success: Pipeline complete! Ready for manuscript writing.
